# AdEx whole-brain inference with VBI 0.4.x

This guarded tutorial validates Brain Act loading and feature extraction by default. Real AdEx simulation and posterior training are opt-in because duration-matched BOLD inference is an HPC workload. See `docs/adex_vbi_inference.md` for the validation requirements.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/tvbtoolkit-matplotlib")
os.environ.setdefault("TVB_USER_HOME", "/tmp/tvbtoolkit-tvb")

from tvbtoolkit import WholeBrainConfig
from tvbtoolkit.inference import (
    AdExBOLDSimulator,
    AdExPrior,
    BOLDFeatureConfig,
    BOLDFeatureExtractor,
    SimulationDataset,
    load_brain_act_bold_mat,
    sample_vbi_posterior,
    simulate_prior,
    train_vbi_posterior,
)


## Load one observation

Set `BRAIN_ACT_MAT` to a subject-level file containing `SC` and `BOLD`. If it is unavailable, the notebook uses a clearly labelled synthetic signal so a fresh clone remains executable.

In [ ]:
default_mat = ROOT / "share/doc_sc_bold_2per_condition_nonsed_chronic_20260720/control/control_c0001.mat"
mat_path = Path(os.environ.get("BRAIN_ACT_MAT", default_mat))
tract_path = ROOT / "data/connectivity/average_aal90/tract_lengths.txt"

if mat_path.exists():
    record = load_brain_act_bold_mat(mat_path, tract_lengths=tract_path)
    bold = record.bold
    sc = record.structural_connectivity
    tract_lengths = record.tract_lengths
    tr_seconds = record.tr_seconds
    observation_label = f"Brain Act: {record.subject_id} ({record.cohort})"
else:
    rng = np.random.default_rng(42)
    tr_seconds = 2.4
    n_time, n_regions = 297, 90
    t = np.arange(n_time) * tr_seconds
    latent = np.column_stack([
        np.sin(2 * np.pi * 0.035 * t),
        np.sin(2 * np.pi * 0.070 * t + 0.5),
        rng.normal(size=n_time),
    ])
    bold = latent @ rng.normal(size=(3, n_regions)) + 0.25 * rng.normal(size=(n_time, n_regions))
    sc = np.loadtxt(ROOT / "data/connectivity/average_aal90/weights.txt")
    sc = sc / np.max(sc)
    tract_lengths = np.loadtxt(tract_path)
    observation_label = "synthetic smoke-test observation (not Brain Act)"

print(observation_label)
print("BOLD:", bold.shape, "SC:", sc.shape, "tract lengths:", tract_lengths.shape)


In [ ]:
feature_extractor = BOLDFeatureExtractor(
    BOLDFeatureConfig(tr_seconds=tr_seconds, n_states=5, state_n_init=10)
)
x_observed = feature_extractor.fit_transform(bold, structural_connectivity=sc)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(np.arange(min(120, bold.shape[0])) * tr_seconds, bold[:120, :6])
axes[0].set(xlabel="Time (s)", ylabel="BOLD", title="Example regional BOLD")
axes[1].imshow(np.corrcoef(bold, rowvar=False), cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("Static FC")
axes[2].bar(np.arange(x_observed.size), x_observed)
axes[2].set(xlabel="Feature index", title=f"Finite inference vector (n={x_observed.size})")
fig.tight_layout()

occupancy = [value for name, value in zip(feature_extractor.feature_names_, x_observed) if name.startswith("state_occupancy_")]
print("all finite:", np.isfinite(x_observed).all(), "occupancy sum:", np.sum(occupancy))


## Declare the prior and matched simulator

`adaptation_b_e` is the explicit AdEx meaning assigned to the requested beta-like parameter. Conduction speed is permitted only because a non-zero, aligned tract-length matrix is attached.

In [ ]:
prior = AdExPrior.default()
for spec in prior.parameters:
    print(f"{spec.name:20s} [{spec.low:g}, {spec.high:g}] {spec.unit} -> {spec.target or spec.model_keys}")

transient_ms = 20_000.0
matched_duration_ms = transient_ms + bold.shape[0] * tr_seconds * 1000.0
base_config = WholeBrainConfig(
    simulation_length_ms=matched_duration_ms,
    dt_ms=0.1,
    zerlaut_order=2,
    stochastic_integrator=True,
    monitor_mode="temporal_average",
    temporal_average_period_ms=1.0,
    weights=sc,
    tract_lengths=tract_lengths,
)
simulator = AdExBOLDSimulator(base_config, prior, feature_extractor, transient_ms=transient_ms)

steps_per_simulation = int(round(matched_duration_ms / base_config.dt_ms))
print(f"Matched duration: {matched_duration_ms / 1000:.1f} s")
print(f"Integration steps per simulation: {steps_per_simulation:,}")
print(f"Steps for a 1,000-simulation campaign: {1000 * steps_per_simulation:,}")


## Generate simulations (opt-in)

Keep this disabled for notebook validation. On HPC, start with prior-predictive QC, then increase the simulation budget and save the resulting dataset.

In [ ]:
RUN_REAL_ADEX = False
NUM_SIMULATIONS = 2
dataset_path = ROOT / "outputs/adex_vbi/simulations.npz"

if RUN_REAL_ADEX:
    dataset = simulate_prior(
        prior,
        simulator,
        num_simulations=NUM_SIMULATIONS,
        feature_names=feature_extractor.feature_names_,
        seed=42,
    )
    dataset.save(dataset_path)
    print("saved", dataset_path, dataset.theta.shape, dataset.features.shape)
else:
    print("Real AdEx simulations are disabled. Set RUN_REAL_ADEX=True on suitable compute.")


## Train and inspect a VBI posterior (opt-in)

Training a posterior from two smoke simulations is invalid. Enable this only after producing an adequate, QC-passed simulation bank.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    dataset = SimulationDataset.load(dataset_path)
    if dataset.theta.shape[0] < 1000:
        raise RuntimeError("Refusing an empirical posterior with fewer than 1,000 simulations.")
    posterior = train_vbi_posterior(dataset, prior, method="SNPE", num_threads=4)
    samples = sample_vbi_posterior(posterior, x_observed, num_samples=10_000)
    fig, axes = plt.subplots(1, prior.ndim, figsize=(3.2 * prior.ndim, 3))
    for index, (axis, name) in enumerate(zip(np.atleast_1d(axes), prior.names)):
        axis.hist(samples[:, index], bins=40, density=True)
        axis.set_title(name)
    fig.tight_layout()
else:
    print("Posterior training is disabled until a sufficient simulation bank exists.")
